**Swin Transformer in One Sentence**

Swin is a Vision Transformer that looks at small local windows, shifts those windows, and gradually makes the image smaller while features become richer.

> **Image** $\rightarrow$ **small patches** $\rightarrow$ **local attention** $\rightarrow$ **shift windows** $\rightarrow$ **merge patches** $\rightarrow$ **repeat** $\rightarrow$ **classify**

---

### 1. Why Swin?

A standard Vision Transformer (ViT) processes a $224 \times 224$ image by having every patch talk to every other patch:

* $\text{👁️ Patch 1} \rightarrow \text{all patches}$
* $\text{👁️ Patch 2} \rightarrow \text{all patches}$
* $\text{👁️ Patch 3} \rightarrow \text{all patches} \dots$

This becomes computationally expensive because self-attention scales quadratically with image size. Swin solves this by restricting communication to nearby neighbors, dividing the image into $7 \times 7$ windows where each window performs attention independently.

> **Feynman Analogy:** Imagine 1,000 people in a room.
> * **ViT:** Everyone talks to everyone at once. 😵
> * **Swin:** People first talk within small local groups, then the groups change so people can talk to new neighbors (shifted windows).
>
>

---

### 2. The Most Important Idea: Shifted Windows

Suppose we have an image grid divided into attention windows represented by letters:

```text
AAAA | BBBB
AAAA | BBBB
-----+-----
CCCC | DDDD
CCCC | DDDD

```

* **Without shifting:** $A$ cannot directly talk to $B$ because they are locked inside separate windows.
* **With shifting:** The model shifts the window boundaries so previously separated patches can interact:

```text
  | AAAA
--+-----
BBBB | CCCC
BBBB | CCCC

```

* **W-MSA:** Attention inside standard windows.
* **SW-MSA:** Attention inside shifted windows.

---

### 3. What Happens to the Image?

Swin is hierarchical, meaning it starts with many small, detailed features and gradually creates fewer but richer features—much like a traditional Convolutional Neural Network (CNN).

| Stage | Spatial Resolution | Channel Dimension |
| --- | --- | --- |
| **Input Patch Embedding** | $56 \times 56$ | $96$ |
| **After Stage 1 (Patch Merging)** | $28 \times 28$ | $192$ |
| **After Stage 2 (Patch Merging)** | $14 \times 14$ | $384$ |
| **After Stage 3 (Patch Merging)** | $7 \times 7$ | $768$ |

> **Easy Memory Trick:** Spatial size $\downarrow$, feature dimension $\uparrow$. Early layers detect basic edges, while later layers recognize complex semantics like a cat's face.

---

### 4. Patch Embedding

Swin breaks the initial $224 \times 224$ image into $4 \times 4$ patches:

$$\frac{224}{4} = 56 \implies 56 \times 56 = 3,136 \text{ patches}$$

Each patch is projected into a vector representation, operating similarly to ViT during the initial tokenization step.

---

### 5. The Swin Block

A standard Swin block processes features through alternating normalization, window attention, and multilayer perceptrons (MLPs) with residual connections:

```text
Input
  ↓
LayerNorm
  ↓
Window Attention (W-MSA or SW-MSA)
  ↓
Residual Connection (+)
  ↓
LayerNorm
  ↓
MLP
  ↓
Residual Connection (+)
  ↓
Output

```

Swin alternates between normal and shifted windows across sequential blocks:

> **Normal window** $\rightarrow$ **Shifted window** $\rightarrow$ **Normal** $\rightarrow$ **Shifted**

---

### 6. What Does the Code Actually Do?

Core operations during a forward pass execute sequentially:

1. **`x = self.norm1(x)`** — Normalizes features.
2. **`torch.roll(...)`** — Performs the cyclic shift.
3. **`window_partition(...)`** — Breaks the feature map into windows.
4. **`self.attn(...)`** — Computes attention inside each window.
5. **`window_reverse(...)`** — Reassembles the windows.
6. **`x = shortcut + ...`** — Adds the residual connection.
7. **`x = x + self.mlp(self.norm2(x))`** — Applies the MLP step.

---

### 7. Why Cyclic Shift?

Cyclic shifting wraps boundaries around so that edge tokens wrap to the opposite side temporarily for window partitioning. This rearranges spatial positioning so that the subsequent attention operation connects different neighboring tokens without requiring expensive global calculations. The shift operation itself contains no learnable weights.

---

### 8. Why Is There an Attention Mask?

After shifting, some tokens end up grouped together inside a window that did not originate from the same local region. An attention mask prevents these artificial neighbors from communicating:

$$\text{Attention Score} + \text{Attention Mask} \xrightarrow{\text{softmax}} \text{Final Weights}$$

Invalid connections receive a heavily negative bias score (e.g., $-100$), which evaluates to approximately zero ($\approx 0$) after the softmax operation.

> **Shift** = create new neighbors; **Mask** = prevent fake cross-window communication.

---

### 9. Window Attention

Standard self-attention computes query-key-value interactions globally:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Swin applies this exact mechanism locally inside small windows. For a $7 \times 7$ window, attention is computed across only $49$ tokens rather than thousands of image patches, yielding massive computational savings.

---

### 10. Relative Position Bias

To help the model understand spatial geometry (e.g., whether a patch is above, below, or beside another), Swin adds a learnable relative position bias directly to the attention matrix:

$$\text{Attention} = QK^T + \text{Relative Position Bias}$$

---

### 11. Patch Merging

After blocks of Swin layers, Patch Merging groups four neighboring patches together ($2 \times 2$ blocks):

```text
A B
C D

```

This merges them into a single token vector $[A, B, C, D]$, cutting spatial resolution in half while doubling the channel depth:

$$H \times W \times C \implies \frac{H}{2} \times \frac{W}{2} \times 2C$$

> **Feynman Explanation:** Four pixels say, *"Let's combine our information into one smarter pixel,"* reducing spatial locations while packing in richer information.

---

### 12. The Entire Swin Architecture

```text
               IMAGE
                 ↓
          Patch Embedding
                 ↓
          56×56 × 96
                 ↓
       ┌─────────────────┐
       │ Swin Blocks     │
       │ W-MSA / SW-MSA  │
       └─────────────────┘
                 ↓
          Patch Merging
                 ↓
          28×28 × 192
                 ↓
       ┌─────────────────┐
       │ Swin Blocks     │
       └─────────────────┘
                 ↓
          Patch Merging
                 ↓
          14×14 × 384
                 ↓
       ┌─────────────────┐
       │ Swin Blocks     │
       └─────────────────┘
                 ↓
          Patch Merging
                 ↓
           7×7 × 768
                 ↓
          Average Pool
                 ↓
           MLP Head
                 ↓
           Prediction

```

---

### 13. Why Call It "Hierarchical"?

Swin bridges the gap between Transformers and CNNs by matching multi-scale processing structures:

* **High Resolution:** Lots of spatial locations, fine-grained low-level features.
* **Lower Resolution:** Fewer locations, large and rich semantic features.

---

### 14. Swin vs. ViT

| Feature | ViT | Swin |
| --- | --- | --- |
| **Attention Scope** | Global attention | Local window attention |
| **Computational Cost** | Expensive for large images | Highly efficient |
| **Architecture Shape** | Flat structure | Hierarchical |
| **Resolution Scaling** | Fixed patch resolution | Progressively downsamples |
| **Patch Interaction** | Every patch can interact | Nearby patches interact locally |
| **Window Shifting** | None | Shifted windows & masking |

> **One-Line Summary:** ViT asks everyone to talk to everyone; Swin makes small groups talk, then changes the groups.

---

### 15. Understand the Code Parameters

* `embed_dim=96` — Initial feature channel dimension.
* `depths=[2, 2, 6, 2]` — Number of Swin blocks stacked in each of the 4 hierarchical stages.
* `num_heads=[3, 6, 12, 24]` — Number of attention heads scaling up across stages.
* `window_size=7` — Dimensions of local attention windows ($7 \times 7$).
* `patch_size=4` — Initial pixel width/height per patch.
* `mlp_ratio=4` — Multiplier expanding the MLP hidden layer dimension ($4 \times \text{embed\_dim}$).
* `drop_path_rate=0.1` — Stochastic depth regularization rate.

---

### 16. The Code You Should Mentally Remember

```python
x = patch_embed(image)

for layer in layers:
    x = SwinBlock(x)       # Alternates W-MSA / SW-MSA
    x = PatchMerging(x)    # Shrinks H,W; grows C

x = norm(x)
x = average_pool(x)
x = head(x)

```

Inside each `SwinBlock`:

```python
x = x + WindowAttention(x)
x = x + MLP(x)

```

---

### 🎯 Feynman Test

* **Why windows?** Reduce attention computation from quadratic to linear complexity relative to image size.
* **Why shift windows?** Enable cross-window communication without global attention overhead.
* **Why mask?** Prevent invalid tokens from interacting after cyclic shifts.
* **Why patch merging?** Downsample spatial resolution while building richer feature dimensions.
* **Why hierarchical?** Combine transformer-based self-attention with CNN-like multi-scale feature pyramids.
* **Swin vs ViT?** ViT uses expensive global attention, whereas Swin uses efficient local shifted-window attention.

> **Final Mental Picture:** Swin is like a city. First, people talk to their immediate neighbors in local blocks. Then, neighborhoods shift so people can meet new adjacent neighbors. Afterward, nearby blocks merge into larger districts with richer shared information. Repeat this hierarchically until the model comprehends the whole city.

In [1]:
# use pretrained swin model for classification

from datasets import load_dataset
from transformers import AutoImageProcessor, SwinForImageClassification
import torch

model=SwinForImageClassification.from_pretrained(
    "microsoft/swin-tiny-patch4-window7-224"
)


image_processor=AutoImageProcessor.from_pretrained(
    "microsoft/swin-tiny-patch4-window7-224"
)


dataset=load_dataset("huggingface/cats-image")
image=dataset["test"]["image"][0]

inputs=image_processor(image,return_tensors="pt")

with torch.no_grad():
  logits=model(**inputs).logits

predicted_label_id=logits.argmax(-1).item()
predicted_label_text=model.config.id2label[predicted_label_id]

print(predicted_label_text)

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

cats_image.jpeg:   0%|          | 0.00/173k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

tabby, tabby cat
